## 환경 세팅

In [ ]:
# ==========================================
# 1. 패키지 의존성 완전 교정 및 설치
# ==========================================
print("1. 파이썬 환경 충돌을 해결하고 패키지를 설치 중입니다... (약 1분 30초 소요)")
!apt-get install -y ffmpeg > /dev/null 2>&1

# 충돌을 일으키는 패키지 삭제 후 검증된 호환 버전 조합으로 강제 설치
!pip uninstall -y numpy torch torchvision torchaudio transformers ctranslate2 whisperx -y > /dev/null 2>&1
!pip install --no-cache-dir "numpy>=1.26.0,<2.0.0" torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 --quiet
!pip install --no-cache-dir transformers ctranslate2 --quiet
!pip install --no-cache-dir git+https://github.com/m-bain/whisperX.git --quiet

import os
import gc
import json
import torch
import whisperx
from google.colab import files

print(">> 패키지 로드 성공! 환경 세팅이 완료되었습니다.")

1. 파이썬 환경 충돌을 해결하고 패키지를 설치 중입니다... (약 1분 30초 소요)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 168.0 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × pip subprocess to install build dependencies did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Installing build dependencies ... error
error: subprocess-exited-with-error

× pip subprocess to install build dependencies did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 205.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.6/39.6 MB 291.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 162.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are inst

## 음절 단위 분석

In [ ]:
# ==========================================
# 2. 오디오 파일 업로드
# ==========================================
print("\n2. 분석할 음성 파일(한국어/영어 지원: mp3, wav, m4a 등)을 업로드해주세요.")
uploaded = files.upload()

if not uploaded:
    raise Exception("파일이 업로드되지 않았습니다. 셀을 다시 실행해주세요.")

audio_file = list(uploaded.keys())[0]
print(f"\n>> 업로드 완료: {audio_file}")

# ==========================================
# 3. WhisperX 변환 & 언어 자동 감지
# ==========================================
device = "cuda" if torch.cuda.is_available() else "cpu"
batch_size = 16
compute_type = "float16" if device == "cuda" else "int8"

print(f"\n3.1 WhisperX 메인 모델을 로드합니다. (장치: {device})")
model = whisperx.load_model("large-v3", device, compute_type=compute_type)

print("\n3.2 1차 음성 인식(STT) 및 언어 자동 감지 진행 중...")
audio = whisperx.load_audio(audio_file)

result = model.transcribe(audio, batch_size=batch_size)

detected_language = result.get("language", "en")
print(f">> 감지된 언어: [{detected_language}]")

# 메모리 정리
del model
gc.collect()
torch.cuda.empty_cache()

# ==========================================
# 4. 언어 맞춤형 Alignment (어절/단어 정렬)
# ==========================================
print(f"\n4. [{detected_language}] 언어 전용 Alignment 모델을 로드하여 어절/단어 타임스탬프를 추출합니다...")

try:
    model_a, metadata = whisperx.load_align_model(language_code=detected_language, device=device)
    aligned_result = whisperx.align(result["segments"], model_a, metadata, audio, device, return_char_alignments=False)

    del model_a
    gc.collect()
    torch.cuda.empty_cache()

except Exception as e:
    print(f"\n[주의] Alignment 처리 중 오류가 발생하여 기본 인식 결과를 사용합니다: {e}")
    aligned_result = result

# ==========================================
# 5. 결과 정리 및 출력
# ==========================================
print("\n" + "="*50)
print(f"        어절/단어 단위 타임스탬프 결과 ({detected_language.upper()})        ")
print("="*50)

word_timestamps = []

for segment in aligned_result["segments"]:
    if "words" in segment:
        for w in segment["words"]:
            word_text = w.get("word", "").strip()
            start_time = w.get("start", None)
            end_time = w.get("end", None)

            if start_time is not None and end_time is not None:
                item = {
                    "word": word_text,
                    "start": round(start_time, 3),
                    "end": round(end_time, 3),
                    "duration": round(end_time - start_time, 3)
                }
                word_timestamps.append(item)
                print(f"[{item['start']:6.3f}s ~ {item['end']:6.3f}s]  {item['word']}")

# ==========================================
# 6. JSON 파일 저장 및 다운로드
# ==========================================
output_json_path = f"{os.path.splitext(audio_file)[0]}_word_timestamps.json"

output_data = {
    "language": detected_language,
    "audio_filename": audio_file,
    "words": word_timestamps
}

with open(output_json_path, "w", encoding="utf-8") as f:
    json.dump(output_data, f, ensure_ascii=False, indent=2)

print("\n" + "="*50)
print(f"결과 저장 완료: {output_json_path}")
print("JSON 결과 파일 다운로드를 시작합니다...")
files.download(output_json_path)


2. 분석할 음성 파일(한국어/영어 지원: mp3, wav, m4a 등)을 업로드해주세요.


Saving [10화 요약] 1년 차 이영X남경에게 벌어진 비상사태🚨 응급 환자의 등장에 멘탈 나간 신입즈😨 ｜ #언젠가는슬기로울전공의생활.mp3 to [10화 요약] 1년 차 이영X남경에게 벌어진 비상사태🚨 응급 환자의 등장에 멘탈 나간 신입즈😨 ｜ #언젠가는슬기로울전공의생활.mp3

>> 업로드 완료: [10화 요약] 1년 차 이영X남경에게 벌어진 비상사태🚨 응급 환자의 등장에 멘탈 나간 신입즈😨 ｜ #언젠가는슬기로울전공의생활.mp3

3.1 WhisperX 메인 모델을 로드합니다. (장치: cuda)
2026-09-01 01:47:46 - whisperx.asr - INFO - No language specified, language will be detected for each audio file (increases inference time)
2026-09-01 01:47:46 - whisperx.vads.pyannote - INFO - Performing voice activity detection using Pyannote...


INFO: Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../usr/local/lib/python3.13/dist-packages/whisperx/assets/pytorch_model.bin`
INFO:lightning.pytorch.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../usr/local/lib/python3.13/dist-packages/whisperx/assets/pytorch_model.bin`



3.2 1차 음성 인식(STT) 및 언어 자동 감지 진행 중...
2026-09-01 01:47:50 - whisperx.asr - INFO - Detected language: ko (1.00) in first 30s of audio
>> 감지된 언어: [ko]

4. [ko] 언어 전용 Alignment 모델을 로드하여 어절/단어 타임스탬프를 추출합니다...


Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]


        어절/단어 단위 타임스탬프 결과 (KO)        
[10.130s ~ 11.131s]  피자
[13.891s ~ 14.171s]  먹고
[14.231s ~ 14.431s]  오지.
[14.571s ~ 14.611s]  이게
[18.392s ~ 31.875s]  더
[31.895s ~ 31.975s]  좋아요.
[35.227s ~ 35.947s]  모레오
[36.307s ~ 36.988s]  선생
[37.068s ~ 37.908s]  나
[37.968s ~ 39.108s]  진짜
[39.628s ~ 40.188s]  헷갈려서
[40.228s ~ 42.769s]  그러는데
[42.929s ~ 43.669s]  비밀로
[44.590s ~ 45.690s]  하고
[45.770s ~ 45.990s]  싶은
[46.390s ~ 47.871s]  거야
[48.611s ~ 48.911s]  소문을
[48.991s ~ 49.131s]  내고
[49.231s ~ 49.311s]  싶은
[49.371s ~ 49.511s]  거야?
[49.551s ~ 54.773s]  모르겠어요.
[54.853s ~ 54.893s]  근데
[54.933s ~ 55.373s]  같이
[55.513s ~ 55.753s]  있고
[55.773s ~ 55.813s]  싶은
[55.833s ~ 55.873s]  건
[55.933s ~ 56.193s]  확실합니다.
[63.507s ~ 63.667s]  잘
[63.787s ~ 63.807s]  안
[63.867s ~ 64.488s]  잡히네.
[64.528s ~ 64.668s]  고기도.
[91.685s ~ 92.065s]  시금치는?
[92.105s ~ 92.586s]  피곤해서
[92.666s ~ 93.126s]  숟가락
[93.386s ~ 93.567s]  들
[93.627s ~ 94.547s]  힘도
[94.607s ~ 96.749s]  없다.
[96.809s ~ 96.889s]  내가
[97.049s ~ 98.411s]  해줘야

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 글자 단위 분석

In [ ]:
# ==========================================
# 2. 오디오 파일 업로드
# ==========================================
print("\n2. 분석할 음성 파일(한국어/영어 지원: mp3, wav, m4a 등)을 업로드해주세요.")
uploaded = files.upload()

if not uploaded:
    raise Exception("파일이 업로드되지 않았습니다. 셀을 다시 실행해주세요.")

audio_file = list(uploaded.keys())[0]
print(f"\n>> 업로드 완료: {audio_file}")

# ==========================================
# 3. WhisperX 변환 & 언어 자동 감지
# ==========================================
device = "cuda" if torch.cuda.is_available() else "cpu"
batch_size = 16
compute_type = "float16" if device == "cuda" else "int8"

print(f"\n3.1 WhisperX 메인 모델을 로드합니다. (장치: {device})")
model = whisperx.load_model("large-v3", device, compute_type=compute_type)

print("\n3.2 1차 음성 인식(STT) 및 언어 자동 감지 진행 중...")
audio = whisperx.load_audio(audio_file)

result = model.transcribe(audio, batch_size=batch_size)

detected_language = result.get("language", "en")
print(f">> 감지된 언어: [{detected_language}]")

# 메모리 정리
del model
gc.collect()
torch.cuda.empty_cache()

# ==========================================
# 4. 글자(Character) 단위 Alignment 정렬
# ==========================================
print(f"\n4. [{detected_language}] 언어 전용 Alignment 모델을 로드하여 '글자 단위' 타임스탬프를 정밀 추출합니다...")

try:
    model_a, metadata = whisperx.load_align_model(language_code=detected_language, device=device)

    # ★ return_char_alignments=True 로 설정하여 글자 단위 데이터를 추출합니다.
    aligned_result = whisperx.align(
        result["segments"],
        model_a,
        metadata,
        audio,
        device,
        return_char_alignments=True
    )

    del model_a
    gc.collect()
    torch.cuda.empty_cache()

except Exception as e:
    print(f"\n[주의] Alignment 처리 중 오류 발생: {e}")
    aligned_result = result

# ==========================================
# 5. 결과 정리 및 글자 단위 타임스탬프 추출
# ==========================================
print("\n" + "="*50)
print(f"        글자(Character) 단위 타임스탬프 결과 ({detected_language.upper()})        ")
print("="*50)

char_timestamps = []

for segment in aligned_result["segments"]:
    # chars 키값에서 글자 단위 타임스탬프를 가져옵니다.
    if "chars" in segment:
        for c in segment["chars"]:
            char_text = c.get("char", "")
            start_time = c.get("start", None)
            end_time = c.get("end", None)

            # 타임스탬프가 정상 측정된 글자만 수집 (공백 포함 여부는 선택 사항)
            if start_time is not None and end_time is not None:
                item = {
                    "char": char_text,
                    "start": round(start_time, 3),
                    "end": round(end_time, 3),
                    "duration": round(end_time - start_time, 3)
                }
                char_timestamps.append(item)
                print(f"[{item['start']:6.3f}s ~ {item['end']:6.3f}s]  '{item['char']}'")

# ==========================================
# 6. JSON 파일 저장 및 다운로드
# ==========================================
output_json_path = f"{os.path.splitext(audio_file)[0]}_char_timestamps.json"

output_data = {
    "language": detected_language,
    "audio_filename": audio_file,
    "characters": char_timestamps
}

with open(output_json_path, "w", encoding="utf-8") as f:
    json.dump(output_data, f, ensure_ascii=False, indent=2)

print("\n" + "="*50)
print(f"결과 저장 완료: {output_json_path}")
print("글자 단위 JSON 결과 파일 다운로드를 시작합니다...")
files.download(output_json_path)


2. 분석할 음성 파일(한국어/영어 지원: mp3, wav, m4a 등)을 업로드해주세요.


Saving [10화 요약] 1년 차 이영X남경에게 벌어진 비상사태🚨 응급 환자의 등장에 멘탈 나간 신입즈😨 ｜ #언젠가는슬기로울전공의생활.mp3 to [10화 요약] 1년 차 이영X남경에게 벌어진 비상사태🚨 응급 환자의 등장에 멘탈 나간 신입즈😨 ｜ #언젠가는슬기로울전공의생활 (1).mp3

>> 업로드 완료: [10화 요약] 1년 차 이영X남경에게 벌어진 비상사태🚨 응급 환자의 등장에 멘탈 나간 신입즈😨 ｜ #언젠가는슬기로울전공의생활 (1).mp3

3.1 WhisperX 메인 모델을 로드합니다. (장치: cuda)
2026-09-01 02:02:12 - whisperx.asr - INFO - No language specified, language will be detected for each audio file (increases inference time)
2026-09-01 02:02:12 - whisperx.vads.pyannote - INFO - Performing voice activity detection using Pyannote...


INFO: Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../usr/local/lib/python3.13/dist-packages/whisperx/assets/pytorch_model.bin`
INFO:lightning.pytorch.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../usr/local/lib/python3.13/dist-packages/whisperx/assets/pytorch_model.bin`



3.2 1차 음성 인식(STT) 및 언어 자동 감지 진행 중...
2026-09-01 02:02:16 - whisperx.asr - INFO - Detected language: ko (1.00) in first 30s of audio
>> 감지된 언어: [ko]

4. [ko] 언어 전용 Alignment 모델을 로드하여 '글자 단위' 타임스탬프를 정밀 추출합니다...


Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]


        글자(Character) 단위 타임스탬프 결과 (KO)        
[10.130s ~ 11.111s]  '피'
[11.111s ~ 11.131s]  '자'
[11.131s ~ 13.891s]  ' '
[13.891s ~ 14.111s]  '먹'
[14.111s ~ 14.171s]  '고'
[14.171s ~ 14.231s]  ' '
[14.231s ~ 14.351s]  '오'
[14.351s ~ 14.411s]  '지'
[14.411s ~ 14.431s]  '.'
[14.431s ~ 14.571s]  ' '
[14.571s ~ 14.591s]  '이'
[14.591s ~ 14.611s]  '게'
[14.611s ~ 18.392s]  ' '
[18.392s ~ 31.875s]  '더'
[31.875s ~ 31.895s]  ' '
[31.895s ~ 31.915s]  '좋'
[31.915s ~ 31.935s]  '아'
[31.935s ~ 31.955s]  '요'
[31.955s ~ 31.975s]  '.'
[35.227s ~ 35.307s]  '모'
[35.307s ~ 35.807s]  '레'
[35.807s ~ 35.947s]  '오'
[35.947s ~ 36.307s]  ' '
[36.307s ~ 36.407s]  '선'
[36.407s ~ 36.988s]  '생'
[36.988s ~ 37.068s]  ' '
[37.068s ~ 37.908s]  '나'
[37.908s ~ 37.968s]  ' '
[37.968s ~ 38.128s]  '진'
[38.128s ~ 39.108s]  '짜'
[39.108s ~ 39.628s]  ' '
[39.628s ~ 39.708s]  '헷'
[39.708s ~ 39.828s]  '갈'
[39.828s ~ 39.928s]  '려'
[39.928s ~ 40.188s]  '서'
[40.188s ~ 40.228s]  ' '
[40.228s ~ 41.449s]  '그'
[41.449s ~ 41.809s]  '러'
[4

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>